In [ ]:
# Install numpy version 1.23.0 for loading the spiral dataset
!pip install numpy==1.23.0

# Install Korean fonts for visualization
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# Restart the session after installing numpy and Korean fonts
# Go to Runtime menu -> Restart runtime


In [ ]:
# Module for using Google Drive
from google.colab import drive
drive.mount('/content/drive')/

In [4]:
import sys
sys.path.append('..')  # Set up to be able to import files from the parent directory
import numpy as np
from common.optimizer import SGD  # Import SGD optimizer from common folder
from dataset import spiral  # Import spiral dataset module
import matplotlib.pyplot as plt  # Import matplotlib for plotting
from two_layer_net import TwoLayerNet  # Import the TwoLayerNet class
plt.rc('font', family='NanumBarunGothic')  # Set NanumBarunGothic as the default font


In [ ]:
# Hyperparameter settings
max_epoch = 300
batch_size = 30
hidden_size = 10
learning_rate = 1.0

# Read data, create model and optimizer
x, t = spiral.load_data()
model = TwoLayerNet(input_size=2, hidden_size=hidden_size, output_size=3)
optimizer = SGD(lr=learning_rate)

# Variables for training
data_size = len(x)
max_iters = data_size // batch_size
total_loss = 0
loss_count = 0
loss_list = []

for epoch in range(max_epoch):
    # Shuffle the data
    idx = np.random.permutation(data_size)
    x = x[idx]
    t = t[idx]

    for iters in range(max_iters):
        batch_x = x[iters*batch_size:(iters+1)*batch_size]
        batch_t = t[iters*batch_size:(iters+1)*batch_size]

        # Compute gradients and update parameters
        loss = model.forward(batch_x, batch_t)
        model.backward()
        optimizer.update(model.params, model.grads)

        total_loss += loss
        loss_count += 1

        # Print training progress periodically
        if (iters+1) % 10 == 0:
            avg_loss = total_loss / loss_count
            print('| Epoch %d | Iteration %d / %d | Loss %.2f'
                  % (epoch + 1, iters + 1, max_iters, avg_loss))
            loss_list.append(avg_loss)
            total_loss, loss_count = 0, 0

# Plot training loss
plt.plot(np.arange(len(loss_list)), loss_list, label='train')
plt.xlabel('Iterations (x10)')
plt.ylabel('Loss')
plt.show()

# Plot decision boundary
h = 0.001
x_min, x_max = x[:, 0].min() - .1, x[:, 0].max() + .1
y_min, y_max = x[:, 1].min() - .1, x[:, 1].max() + .1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
X = np.c_[xx.ravel(), yy.ravel()]
score = model.predict(X)
predict_cls = np.argmax(score, axis=1)
Z = predict_cls.reshape(xx.shape)
plt.contourf(xx, yy, Z)
plt.axis('off')

# Plot data points
x, t = spiral.load_data()
N = 100
CLS_NUM = 3
markers = ['o', 'x', '^']
for i in range(CLS_NUM):
    plt.scatter(x[i*N:(i+1)*N, 0], x[i*N:(i+1)*N, 1], s=40, marker=markers[i])
plt.show()
